<a href="https://colab.research.google.com/github/grsart/BiomolComp/blob/main/P11/03_TFBinding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sítio de Interação de Fator de Transcrição


Fatores de transcrição são tipicamente proteínas que reconhecem sequências específicas, e de tamanho específico, do DNA e regulam a transcrição de genes nas adjacências do sítio de interação.

Os sítios de interação tipicamente contem ~10 bp com uma assinatura de sequência mais ou menos conhecida (e.g.: AGGTCA).

## Fator de Transcrição JUND

As sequências de DNA que são reconhecidas pelo fator de transcrição JUND foram mapeadas exaustivamente no genoma humano. Neste experimento, vamos usar apenas os dados do cromossomo 22 (um dos menores), e que conta com 50 milhões de bases. Neste experimento, elas foram divididas em segmentos de 101 pares de bases e foram rotuladas para indicar a ligação ou não do JUND a este segmento.

Novamente, vamos usar o *one-hot encoded* para representar as sequências: vetores de quatro elementos contendo um 1 e três 0, para representar A,G,C ou T.

**Q1: A representação em um vetor one-hot encode de quatro elementos é suficiente para todos os casos? Como resolveria situações onde aparece X na representação?**

Novamente vamos usar uma rede CNN, mas desta vez uma rede 1D, já que o dado é também 1D.

## Bibliotecas

Esta versão **não usa o DeepChem**: os dados do livro são lidos diretamente dos arquivos
`.joblib` e o modelo é construído com o Keras que já vem no Colab. Menos uma camada de
dependências para quebrar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU') or "nenhuma (CPU)")

## Carregando os dados

In [ ]:
import os
if not os.path.isdir('DeepLearningLifeSciences'):
    !git clone --depth 1 -q https://github.com/deepchem/DeepLearningLifeSciences.git
%cd -q DeepLearningLifeSciences/Chapter06
!ls

In [ ]:
import joblib, numpy as np

def carregar(pasta):
    """Lê um conjunto do DeepChem direto dos arquivos .joblib, sem precisar da biblioteca.
    X = características, y = rótulo, w = peso de cada amostra."""
    X = joblib.load(f"{pasta}/shard-0-X.joblib")
    y = joblib.load(f"{pasta}/shard-0-y.joblib")
    w = joblib.load(f"{pasta}/shard-0-w.joblib")
    return np.asarray(X, "float32"), np.asarray(y, "float32"), np.asarray(w, "float32")

X_tr, y_tr, w_tr = carregar('train_dataset')
X_va, y_va, w_va = carregar('valid_dataset')

print("treino:   X", X_tr.shape, " y", y_tr.shape)
print("validação: X", X_va.shape, " y", y_va.shape)
print("\nFração de segmentos que ligam JUND: %.4f (treino), %.4f (validação)" %
      (y_tr.mean(), y_va.mean()))
print("Ou seja: cerca de 1 sítio positivo a cada %d segmentos." % round(1 / y_tr.mean()))

**Q1: A representação one-hot de quatro elementos é suficiente para todos os casos? Como você
resolveria as posições em que o genoma traz um `N` (base não determinada)?**

**Q2: Olhe a fração de positivos impressa acima. Se um modelo preguiçoso respondesse "não liga"
para todos os segmentos, qual seria a acurácia dele? Isso o tornaria útil?**

### Pesos das amostras

O conjunto vem com um vetor `w` de pesos. Ele existe justamente por causa do desbalanceamento: os
poucos segmentos positivos recebem peso maior, para que o erro cometido neles pese tanto quanto o
erro cometido no mar de negativos. O Keras aceita isso diretamente em `sample_weight`.

In [ ]:
print("pesos distintos:", np.unique(w_tr.round(3))[:6], "...")
print("peso médio dos positivos:", w_tr[y_tr == 1].mean().round(3))
print("peso médio dos negativos:", w_tr[y_tr == 0].mean().round(3))

## Construção do modelo

Convolução **1D**, porque a sequência é unidimensional. A entrada tem forma 101 x 4.

**Q3: Explique o que há no vetor de características de tamanho 101 x 4.**

**Q4: Qual a função de ativação das camadas ocultas? E a da camada de saída? Por quê?**

**Q5: Quantas camadas ocultas tem o modelo? Qual a largura de cada uma?**

**Q6: Que tipo de regularização é usada abaixo? Qual o princípio e o que significa o valor 0.5?**

**Q7: Qual a função de perda? Por que ela, e não o erro quadrático médio?**

In [ ]:
modelo = keras.Sequential([
    keras.Input(shape=(101, 4)),
    layers.Conv1D(15, kernel_size=10, activation='relu', padding='same'),
    layers.Dropout(0.5),
    layers.Conv1D(15, kernel_size=10, activation='relu', padding='same'),
    layers.Dropout(0.5),
    layers.Conv1D(15, kernel_size=10, activation='relu', padding='same'),
    layers.Dropout(0.5),
    layers.Flatten(),
    layers.Dense(1, activation='sigmoid'),
])
modelo.compile(optimizer='adam', loss='binary_crossentropy', weighted_metrics=['accuracy'])
modelo.summary()

## Treinamento

**Q8: O que significa rodar 10 épocas?**


In [ ]:
parada = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                       restore_best_weights=True)

historico = modelo.fit(X_tr, y_tr, sample_weight=w_tr,
                       validation_data=(X_va, y_va, w_va),
                       epochs=30, batch_size=128,
                       callbacks=[parada], verbose=2)

## Curvas de treinamento

In [ ]:
h = historico.history
plt.figure(figsize=(6, 4))
plt.plot(h['loss'], '-o', label='treino')
plt.plot(h['val_loss'], '-o', label='validação')
plt.xlabel('época'); plt.ylabel('perda (entropia cruzada ponderada)')
plt.legend(); plt.title('Treinamento'); plt.tight_layout(); plt.show()
print("melhor época:", int(np.argmin(h['val_loss'])) + 1)

## Métricas: por que a AUC da ROC não basta aqui

A ROC usa a taxa de falsos positivos, cujo denominador são os **negativos** — e aqui eles são a
maioria esmagadora. Com classes muito desbalanceadas, a curva ROC fica otimista. A curva
**precisão-revocação** mostra o problema de forma mais honesta: ela ignora os verdadeiros negativos.

A linha de base da precisão-revocação não é 0,5: é a própria fração de positivos.

In [ ]:
p_va = modelo.predict(X_va, verbose=0).ravel()
p_tr = modelo.predict(X_tr, verbose=0).ravel()

print("ROC AUC  [treino]:    %.3f" % roc_auc_score(y_tr, p_tr))
print("ROC AUC  [validação]: %.3f" % roc_auc_score(y_va, p_va))
print("PR  AUC  [validação]: %.3f   (acaso = %.3f)" %
      (average_precision_score(y_va, p_va), y_va.mean()))

fpr, tpr, _ = roc_curve(y_va, p_va)
prec, rec, _ = precision_recall_curve(y_va, p_va)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(fpr, tpr); a1.plot([0, 1], [0, 1], '--', color='gray')
a1.set_xlabel('1 - especificidade (FPR)'); a1.set_ylabel('sensibilidade (TPR)'); a1.set_title('ROC')
a2.plot(rec, prec); a2.axhline(y_va.mean(), ls='--', color='gray')
a2.set_xlabel('revocação (sensibilidade)'); a2.set_ylabel('precisão'); a2.set_title('Precisão-Revocação')
plt.tight_layout(); plt.show()

**Q9: Interprete as curvas de perda. A partir de que época a validação deixa de melhorar?**

**Q10: Compare a AUC da ROC com a AUC da precisão-revocação. Por que os dois números são tão
diferentes? Qual deles você reportaria para um biólogo que vai validar os sítios na bancada?**

**Q11 (opcional): o arquivo `accessibility.txt` do mesmo capítulo traz a acessibilidade da cromatina
para cada segmento. Como você acrescentaria essa informação ao modelo, e por que ela deveria ajudar?**